# 27 - Meta-Prompt Generator with LLM-as-Judge Evaluation
**Stack:** LangChain LCEL · Python 3.12 · OpenAI API
**Adapted from:** Meta-prompt generator with validation pipeline

A LangChain pipeline that automatically generates prompt variants,
tests them, and scores results with an LLM judge.

**Agent OS relevance:** This is a compact, complete example of the evaluation
infrastructure pattern -- LLM-as-judge, structured scoring, regression tracking.
The same judge pattern underpins agent output evaluation.

Concepts covered:
- What is a meta-prompt?
- Chain-of-thought and why it matters
- LLM-as-judge: scoring approach, bias taxonomy, mitigations
- Production deployment considerations

In [ ]:
# Install: pip install langchain>=0.3 langchain-core langchain-openai python-dotenv

import json, os
from dataclasses import asdict, dataclass
from typing import List, Optional
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Load API key from environment or .env file
# from dotenv import load_dotenv; load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', 'MOCK_MODE')
NUM_VARIANTS   = 4
MOCK_MODE      = OPENAI_API_KEY == 'MOCK_MODE'

print(f'Mode: {"mock" if MOCK_MODE else "live"} | Variants: {NUM_VARIANTS}')

## 1. What is a Meta-Prompt?

A **prompt** is the text you send to an LLM.
A **meta-prompt** is a prompt that generates other prompts.

Why? Because identical intent can be phrased in many ways, and LLMs are
sensitive to that phrasing. A meta-prompt explores the space of valid phrasings
automatically.

Four meta-prompt patterns:
1. **Variant generation** - 'Rewrite this prompt 5 ways'
2. **Self-improvement** - 'Here is my prompt. Make it better. Explain why.'
3. **Decomposition** - 'Break this complex prompt into steps'
4. **Role assignment** - 'Rewrite for an expert audience vs. a novice'

In [ ]:
@dataclass
class PromptResult:
    variant: str
    response: str
    score: float
    rationale: str

# Prompt templates
VARIANT_PROMPT = ChatPromptTemplate.from_messages([
    ('system', 'You are an expert prompt engineer. Generate {n} distinct rewrites of the given prompt. Return a JSON array of strings only.'),
    ('human', 'Base prompt: {base_prompt}'),
])

WORKER_PROMPT = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful customer service agent for a field-service company.'),
    ('human', '{variant}'),
])

JUDGE_PROMPT = ChatPromptTemplate.from_messages([
    ('system', '''You are an expert evaluator. Score this (prompt, response) pair from 0.0 to 1.0.
Success intent: {intent}
Return JSON only: {"score": 0.0-1.0, "rationale": "brief explanation"}'''),
    ('human', 'Prompt: {variant}\n\nResponse: {response}'),
])

print('Prompt templates defined')

## 2. Chain-of-Thought

Chain-of-thought (CoT) is prompting an LLM to reason step-by-step before
giving a final answer. Classic form: 'Let\'s think step by step.'

**Why it matters here:**
- The judge is asked for a `rationale` alongside its score -- that IS chain-of-thought
- The rationale makes the score auditable and debuggable
- Modern reasoning models (o1, claude-3-7-sonnet) use extended CoT internally

**CoT in the judge rubric:**
Asking for rationale before score improves judge consistency by ~15-20%
(the judge 'thinks' before committing to a number).

In [ ]:
def mock_variant_generator(base_prompt: str, n: int) -> list[str]:
    templates = [
        f'{base_prompt}',
        f'Please {base_prompt.lower()}',
        f'I need to {base_prompt.lower()}. Can you help?',
        f'Could you {base_prompt.lower()}? Thank you.',
        f'Help me with this: {base_prompt}',
    ]
    return templates[:n]

def mock_worker(variant: str) -> str:
    responses = {
        'check order status': 'Your order #12345 is currently in transit and will arrive in 2 business days.',
    }
    base = 'check order status'
    if any(w in variant.lower() for w in ['order', 'status', 'check']):
        return f'Order status: Your most recent order is in transit. Expected delivery: 2 business days. Tracking: TRK-{hash(variant) % 99999:05d}'
    return f'I can help with that. Please provide your order number.'

def mock_judge(variant: str, response: str, intent: str) -> dict:
    score = 0.9 if 'order' in response.lower() and 'tracking' in response.lower() else 0.6
    if len(response) < 20:
        score -= 0.2
    return {'score': round(score, 2), 'rationale': f'Response addresses intent ({intent}): {"yes" if score > 0.7 else "partially"}'}

print('Mock pipeline functions defined')

## 3. LLM-as-Judge: Scoring, Bias, Mitigations

**Known biases:**
- Verbosity bias: judges reward longer answers regardless of quality
- Position bias: in pairwise comparison, first option often preferred
- Self-preference: judge favors outputs from its own model family
- Score compression: judges drift toward the middle without anchor examples

**Mitigations used here:**
- Score independently (not pairwise) to avoid position bias
- Use a stronger model as judge than worker
- Require rationale (CoT) to reduce score compression
- Calibrate against human labels on a small reference set

In [ ]:
def run_pipeline(base_prompt: str, intent: str, n_variants: int = NUM_VARIANTS) -> list[PromptResult]:
    print(f'Base prompt: {base_prompt!r}')
    print(f'Generating {n_variants} variants...\n')

    if MOCK_MODE:
        variants  = mock_variant_generator(base_prompt, n_variants)
        responses = [mock_worker(v) for v in variants]
        judgments = [mock_judge(v, r, intent) for v, r in zip(variants, responses)]
    else:
        from langchain_openai import ChatOpenAI
        gen_llm    = ChatOpenAI(model='gpt-4o-mini', temperature=0.9)
        worker_llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.3)
        judge_llm  = ChatOpenAI(model='gpt-4o',      temperature=0.0)
        gen_chain    = VARIANT_PROMPT | gen_llm | JsonOutputParser()
        worker_chain = WORKER_PROMPT  | worker_llm | StrOutputParser()
        judge_chain  = JUDGE_PROMPT   | judge_llm  | JsonOutputParser()
        variants  = gen_chain.invoke({'base_prompt': base_prompt, 'n': n_variants})
        responses = [worker_chain.invoke({'variant': v}) for v in variants]
        judgments = [judge_chain.invoke({'variant': v, 'response': r, 'intent': intent})
                     for v, r in zip(variants, responses)]

    results = [
        PromptResult(variant=v, response=r, score=j['score'], rationale=j['rationale'])
        for v, r, j in zip(variants, responses, judgments)
    ]
    return sorted(results, key=lambda x: x.score, reverse=True)

results = run_pipeline(
    base_prompt='Check order status',
    intent='Customer should receive current order status including tracking info',
)

print(f'{'Rank':<5} {'Score':<8} Variant (first 50 chars)')
print('-' * 70)
for i, r in enumerate(results, 1):
    print(f'{i:<5} {r.score:<8.2f} {r.variant[:50]}')
    print(f'       Rationale: {r.rationale}')
    print()

In [ ]:
# Save results for regression tracking
import json, time
run_record = {
    'run_id': f'run_{int(time.time())}',
    'base_prompt': 'Check order status',
    'results': [asdict(r) for r in results],
    'best_score': results[0].score if results else 0,
    'mean_score': sum(r.score for r in results) / len(results) if results else 0,
}
with open('/tmp/meta_prompt_results.json', 'w') as f:
    json.dump(run_record, f, indent=2)

print(f'Results saved to /tmp/meta_prompt_results.json')
print(f'Best score:  {run_record["best_score"]:.2f}')
print(f'Mean score:  {run_record["mean_score"]:.2f}')

## 4. Production Deployment

**Observability:** Wrap each chain step with LangSmith or self-hosted Langfuse
to get per-step latency, token counts, and trace IDs.

**Cost control:** Run variant generation with a cheap model (gpt-4o-mini),
reserve judge calls for the top-K candidates only.

**Regression gate:** In CI, run this eval suite on every prompt change.
Fail the build if mean_score drops > 0.05 from baseline.

**Air-gapped deployment:** Swap `ChatOpenAI` for `ChatOllama`:
```python
from langchain_ollama import ChatOllama
llm = ChatOllama(model='gpt-oss:20b', base_url='http://localhost:11434')
```
No network egress required.